# 2 · STL Generation

**Author:** Héctor Fernández Pinacho  
**Lab:** IDEAL Lab · ETH Zürich

Turns each sampled propeller configuration into a 3-D mesh. Every row of the geometry table is sent to RhinoCompute, which drives the parametric Grasshopper definition of the Propeller Configurator and returns the finished propeller mesh; the mesh is written as a binary STL, its enclosed volume is computed, and a validity flag records whether the STL is a usable single solid. This notebook requires Rhinoceros to be installed; the path to `rhino.compute.exe` may need to be adjusted in the configuration cell.

**Physics inputs:** `csv/01_geometry.csv` — the 17 geometry parameters of each configuration

**Physics outputs:** `stl/prop_<id>.stl` — one mesh per configuration; `csv/02_stl_volumes.csv` — enclosed volume [L, m³] and the `stl_ok` / `single_solid` validity flags

**Structure:**

1. Imports
2. Configuration — all tunable constants, paths and settings
3. Function definitions — RhinoCompute interface, STL writing, volume and validity checks
4. Main code — server start, batch generation, volume calculation and export


## 1. Imports

In [ ]:
import os
import sys

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True

import json
import struct
import subprocess
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import rhino3dm
import trimesh

import pipeline_config as cfg
from compute_rhino3d import Grasshopper, Util
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from tqdm.auto import tqdm

## 2. Configuration

In [ ]:
RHINO_COMPUTE_URL = cfg.RHINO_COMPUTE_URL
RHINO_COMPUTE_API_KEY = cfg.RHINO_COMPUTE_API_KEY
RHINO_COMPUTE_EXE = os.path.expandvars(cfg.RHINO_COMPUTE_EXE)

BASE_DIR = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
UTILS_DIR = BASE_DIR / 'utils'
GH_FILE = UTILS_DIR / cfg.GH_FILE_NAME
CSV_DIR = BASE_DIR / 'csv'
STL_DIR = BASE_DIR / 'stl'
INPUT_CSV = CSV_DIR / cfg.CSV_NAMES['geometry']
OUTPUT_CSV = CSV_DIR / cfg.CSV_NAMES['stl_volumes']

MAX_CONFIGS = cfg.NB2_MAX_CONFIGS
MAX_WORKERS = min(os.cpu_count(), cfg.NB2_MAX_WORKERS_CAP)
GENERATION_PASSES = cfg.GENERATION_PASSES
RUN_SINGLE_TEST_FIRST = cfg.RUN_SINGLE_TEST_FIRST
TEST_CONFIG_INDEX = cfg.TEST_CONFIG_INDEX

EXPORT_SOLUTION = cfg.EXPORT_SOLUTION
POSITION_OFFSET = cfg.POSITION_OFFSET
STL_OK_MIN_L = cfg.STL_OK_MIN_L

CSV_TO_GH = cfg.CSV_TO_GH

for required_path in [GH_FILE, INPUT_CSV]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

STL_DIR.mkdir(parents=True, exist_ok=True)

params_df = pd.read_csv(INPUT_CSV)
if MAX_CONFIGS:
    params_df = params_df.head(MAX_CONFIGS).copy()

print(f'Configurations : {len(params_df)}')
print(f'GH definition  : {GH_FILE}')
print(f'Output CSV     : {OUTPUT_CSV}')

## 3. Function Definitions

- **rhinocompute_is_alive(url, timeout)** — pings the local RhinoCompute server and returns True if it answers, so the notebook knows whether the geometry engine is running before sending it work.
- **row_to_gh_inputs(row)** — converts one propeller's CSV parameters into the named inputs the Grasshopper definition expects. Takes a geometry-table row and returns a dictionary of Grasshopper input names and values.
- **call_rhinocompute(gh_path, inputs)** — packs those inputs into Grasshopper data trees, sends them to the RhinoCompute server, and returns the decoded JSON response containing the resulting meshes.
- **generate_stl(row)** — the full per-propeller step: maps the row, calls RhinoCompute, decodes the returned meshes and writes `stl/prop_<id>.stl`. Skips configurations whose STL already exists. Returns a record with the config id, an `stl_ok` flag and an error message when generation failed.
- **iterate_mesh_triangles(mesh)** — walks through every face of a Rhino mesh, splits quads into two triangles, and returns the list of valid triangle vertex-index triples.
- **vertex_xyz(v)** — returns the x, y, z coordinates of a single mesh vertex regardless of the vertex object type.
- **write_stl_file(meshes, filepath)** — writes a list of Rhino meshes to a binary STL file on disk, computing the unit normal of every triangle.
- **stl_volume_cm3(stl_path)** — reads a finished STL and returns its enclosed solid volume in cubic centimetres (used later to estimate mass), or None if the mesh cannot be read.
- **stl_shell_count(stl_path, weld_decimals)** — reads a binary STL directly, welds coincident vertices, builds the triangle-vertex connectivity graph and returns how many separate connected shells the mesh has.
- **stl_is_single_solid(stl_path)** — returns True only if the STL is exactly one connected shell.
- **stl_path_for_config(config_id)** — returns the canonical STL path `stl/prop_<id>.stl` for a configuration.


In [ ]:
def rhinocompute_is_alive(url, timeout=2.0):

    try:
        urllib.request.urlopen(url, timeout=timeout)
        alive = True
    except Exception:
        alive = False

    return alive


def row_to_gh_inputs(row):

    missing = []
    for csv_column in CSV_TO_GH:
        if csv_column not in row.index:
            missing.append(csv_column)
    if missing:
        raise KeyError(f'Missing CSV columns: {missing}')
    inputs = {}
    for csv_column, gh_name in CSV_TO_GH.items():
        inputs[gh_name] = float(row[csv_column])
    inputs['positionOffset'] = int(POSITION_OFFSET)
    inputs['exportSolution'] = bool(EXPORT_SOLUTION)
    inputs['sheetId'] = '\u200b'

    return inputs


def call_rhinocompute(gh_path, inputs):

    trees = []
    for name, value in inputs.items():
        tree = Grasshopper.DataTree(name)
        tree.Append([0], [value])
        trees.append(tree)
    raw = Grasshopper.EvaluateDefinition(str(gh_path), trees)
    if isinstance(raw, str):
        response = json.loads(raw)
    else:
        response = raw

    return response


def generate_stl(row):

    config_id = int(row['config_id'])
    stl_path = STL_DIR / f'prop_{config_id}.stl'
    record = dict(config_id=config_id, stl_ok=False, error_message='')

    if stl_path.exists() and stl_path.stat().st_size > 84:
        record['stl_ok'] = True

        return record

    try:
        inputs = row_to_gh_inputs(row)
        response = call_rhinocompute(GH_FILE, inputs)
        values = {}
        for value_entry in response.get('values', []):
            values[value_entry['ParamName']] = value_entry

        if 'MeshFinal' not in values:
            record['error_message'] = f"'MeshFinal' not found in GH outputs: {list(values.keys())}"

            return record

        meshes = []
        for branch in (values['MeshFinal'].get('InnerTree') or {}).values():
            for item in branch:
                raw = item.get('data')
                if not isinstance(raw, str):
                    continue
                stripped = raw.strip().strip('"')
                obj = None
                if stripped.startswith('RFJBQ08') or stripped.startswith('DRACO'):
                    try:
                        obj = rhino3dm.DracoCompression.DecompressBase64String(stripped)
                    except Exception:
                        obj = None
                else:
                    try:
                        obj = rhino3dm.CommonObject.Decode(json.loads(raw))
                    except Exception:
                        obj = None
                if isinstance(obj, rhino3dm.Mesh):
                    meshes.append(obj)

        if not meshes:
            errors = []
            for error_entry in (response.get('errors') or []):
                errors.append(error_entry.get('Message', str(error_entry)))
            record['error_message'] = f'No mesh in response. GH errors: {errors}'

            return record

        write_stl_file(meshes, stl_path)
        record['stl_ok'] = True
    except Exception as exc:
        record['error_message'] = str(exc)

    return record


def iterate_mesh_triangles(mesh):

    vertices = mesh.Vertices
    vertex_count = len(vertices)
    triangles = []
    for face in mesh.Faces:
        try:
            a = int(face[0])
            b = int(face[1])
            c = int(face[2])
            d = int(face[3])
        except Exception:
            a = int(face.A)
            b = int(face.B)
            c = int(face.C)
            d = int(getattr(face, 'D', face.C))
        if min(a, b, c) < 0 or max(a, b, c) >= vertex_count:
            continue
        triangles.append((a, b, c))
        if d != c and 0 <= d < vertex_count:
            triangles.append((a, c, d))

    return triangles


def vertex_xyz(v):

    if hasattr(v, 'X'):
        coordinates = (float(v.X), float(v.Y), float(v.Z))
    else:
        coordinates = (float(v[0]), float(v[1]), float(v[2]))

    return coordinates


def write_stl_file(meshes, filepath):

    mesh_triangle_pairs = []
    total_triangles = 0
    for mesh in meshes:
        triangles = iterate_mesh_triangles(mesh)
        mesh_triangle_pairs.append((mesh, triangles))
        total_triangles += len(triangles)

    with open(filepath, 'wb') as stl_file:
        stl_file.write(b'STL by 2_stl_generation.ipynb'.ljust(80, b'\x00'))
        stl_file.write(struct.pack('<I', total_triangles))
        for mesh, faces in mesh_triangle_pairs:
            vertices = mesh.Vertices
            for a, b, c in faces:
                p0 = vertex_xyz(vertices[a])
                p1 = vertex_xyz(vertices[b])
                p2 = vertex_xyz(vertices[c])
                ux = p1[0] - p0[0]
                uy = p1[1] - p0[1]
                uz = p1[2] - p0[2]
                vx = p2[0] - p0[0]
                vy = p2[1] - p0[1]
                vz = p2[2] - p0[2]
                nx = uy * vz - uz * vy
                ny = uz * vx - ux * vz
                nz = ux * vy - uy * vx
                normal_length = (nx ** 2 + ny ** 2 + nz ** 2) ** 0.5
                if normal_length > 0:
                    nx = nx / normal_length
                    ny = ny / normal_length
                    nz = nz / normal_length
                stl_file.write(struct.pack('<fff', nx, ny, nz))
                stl_file.write(struct.pack('<fff', *p0))
                stl_file.write(struct.pack('<fff', *p1))
                stl_file.write(struct.pack('<fff', *p2))
                stl_file.write(struct.pack('<H', 0))

In [ ]:
def stl_volume_cm3(stl_path):

    try:
        mesh = trimesh.load(str(stl_path), force='mesh')
        volume = abs(float(mesh.volume))
    except Exception:
        volume = None

    return volume


def stl_shell_count(stl_path, weld_decimals=5):

    try:
        with open(stl_path, 'rb') as stl_file:
            stl_file.read(80)
            triangle_count = struct.unpack('<I', stl_file.read(4))[0]
            raw = np.frombuffer(stl_file.read(triangle_count * 50), dtype=np.uint8)
        raw = raw.reshape(triangle_count, 50)
        triangle_vertices = raw[:, :48].view('<f4').reshape(triangle_count, 12)[:, 3:12].reshape(triangle_count, 3, 3)
    except Exception:

        return 0

    if triangle_count == 0:

        return 0

    rounded = np.round(triangle_vertices.reshape(-1, 3), weld_decimals)
    welded_vertices, inverse = np.unique(rounded, axis=0, return_inverse=True)
    triangle_vertex_ids = inverse.reshape(triangle_count, 3)
    vertex_count = len(welded_vertices)

    triangle_nodes = np.repeat(np.arange(triangle_count), 3)
    vertex_nodes = triangle_vertex_ids.reshape(-1) + triangle_count
    rows = np.concatenate([triangle_nodes, vertex_nodes])
    cols = np.concatenate([vertex_nodes, triangle_nodes])
    total_nodes = triangle_count + vertex_count
    adjacency = coo_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(total_nodes, total_nodes)).tocsr()
    component_count, node_labels = connected_components(adjacency, directed=False)
    triangle_labels = node_labels[:triangle_count]

    return len(np.unique(triangle_labels))


def stl_is_single_solid(stl_path):

    return stl_shell_count(stl_path) == 1


def stl_path_for_config(config_id):

    return STL_DIR / f'prop_{int(config_id)}.stl'

## 4. Main Code

The main code fixes the RhinoCompute settings and paths, loads the geometry table, starts (or connects to) the RhinoCompute server, generates every missing STL in a parallel batch, computes the enclosed volume of each mesh, and finally writes `csv/02_stl_volumes.csv` with the `stl_ok` flag. A configuration is accepted (`stl_ok = True`) only when its STL file exists, its volume is at least the minimum printable volume, and the mesh is a single connected solid.


### 4.1 Start RhinoCompute

In [ ]:
if rhinocompute_is_alive(RHINO_COMPUTE_URL):
    print(f'RhinoCompute already running at {RHINO_COMPUTE_URL}')
    rhinocompute_process = None
else:
    if not Path(RHINO_COMPUTE_EXE).exists():
        raise FileNotFoundError(f'rhino.compute.exe not found:\n  {RHINO_COMPUTE_EXE}\n' 'Update RHINO_COMPUTE_EXE in the configuration cell or start it manually.')
    print('Starting RhinoCompute...')
    rhinocompute_process = subprocess.Popen([RHINO_COMPUTE_EXE], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for test_index in range(30):
        time.sleep(1)
        if rhinocompute_is_alive(RHINO_COMPUTE_URL):
            print(f'RhinoCompute ready after {test_index + 1}s (PID {rhinocompute_process.pid})')
            break
    else:
        raise RuntimeError('RhinoCompute did not respond within 30s.')

Util.url = RHINO_COMPUTE_URL
Util.apiKey = RHINO_COMPUTE_API_KEY

### 4.2 STL Generation

Runs one test configuration first to verify the Grasshopper connection, then generates every missing STL in parallel. Configurations whose `prop_<id>.stl` already exists are skipped, so the batch is restartable at any time. RhinoCompute drops a small percentage of solves transiently under a long batch, so the loop makes up to `GENERATION_PASSES` passes — each pass retries only the configurations that still have no STL — and stops early once the failure count stops improving (the remaining few configs fail deterministically inside the Grasshopper definition).


In [ ]:
if RUN_SINGLE_TEST_FIRST:
    print('=' * 60)
    test = generate_stl(params_df.iloc[TEST_CONFIG_INDEX])
    for key, value in test.items():
        print(f'  {key:<20}: {value}')
    if not test['stl_ok']:
        raise RuntimeError('Single test failed — fix GH definition before running batch.')
    print('=' * 60)

rows = []
for row_index, row in params_df.iterrows():
    rows.append(row)

previous_fail_count = None
for generation_pass in range(1, GENERATION_PASSES + 1):
    results = [None] * len(rows)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {}
        for i, row in enumerate(rows):
            futures[pool.submit(generate_stl, row)] = i
        with tqdm(total=len(rows), desc=f'Generating STLs, pass {generation_pass} (workers={MAX_WORKERS})') as pbar:
            for future in as_completed(futures):
                i = futures[future]
                results[i] = future.result()
                pbar.update(1)

    results_df = pd.DataFrame(results)
    n_ok = int(results_df['stl_ok'].sum())
    n_fail = len(results_df) - n_ok
    print(f'Pass {generation_pass}: {n_ok} STLs present, {n_fail} failed')
    if n_fail == 0:
        break
    if previous_fail_count is not None and n_fail >= previous_fail_count:
        print('No further progress between passes — the remaining configs fail deterministically in Grasshopper.')
        break
    previous_fail_count = n_fail

print(f'Done — {n_ok} STLs present, {n_fail} failed')

### 4.3 Volume Calculation

Loads the existing volumes CSV when present and computes only the missing volumes, so the notebook can be re-run incrementally.


In [ ]:
if OUTPUT_CSV.exists():
    existing_df = pd.read_csv(OUTPUT_CSV)
    results_df = results_df.merge(existing_df[['config_id', 'volume_L', 'volume_m3']], on='config_id', how='left')
    ids_needing_volume = set(results_df.loc[results_df['stl_ok'] & results_df['volume_L'].isna(), 'config_id'])
    print(f'Existing CSV found — {len(existing_df)} rows loaded.')
    print(f'IDs needing volume calculation: {len(ids_needing_volume)}')
else:
    results_df['volume_L'] = None
    results_df['volume_m3'] = None
    ids_needing_volume = set(results_df.loc[results_df['stl_ok'], 'config_id'])
    print(f'No existing CSV — computing volumes for all {len(ids_needing_volume)} props.')

to_compute = results_df[results_df['config_id'].isin(ids_needing_volume)]
for idx, row in tqdm(to_compute.iterrows(), total=len(to_compute), desc='Volume'):
    stl_path = STL_DIR / f"prop_{int(row['config_id'])}.stl"
    vol_mm3 = stl_volume_cm3(stl_path)
    if vol_mm3 is not None:
        results_df.at[idx, 'volume_L'] = vol_mm3 / 1e6
        results_df.at[idx, 'volume_m3'] = vol_mm3 / 1e9
    else:
        results_df.at[idx, 'volume_L'] = None
        results_df.at[idx, 'volume_m3'] = None

vols = results_df['volume_L'].dropna()
n_vol = len(vols)
n_nan = results_df['volume_L'].isna().sum()
mean_val = vols.mean()
id_min = int(results_df.loc[vols.idxmin(), 'config_id'])
id_max = int(results_df.loc[vols.idxmax(), 'config_id'])
id_mean = int(results_df.loc[(vols - mean_val).abs().idxmin(), 'config_id'])

print(f'Volumes computed: {n_vol}/{len(results_df)}')
print(f'  mean  {mean_val:.4f} L  (config_id {id_mean})')
print(f'  min   {vols.min():.4f} L  (config_id {id_min})')
print(f'  max   {vols.max():.4f} L  (config_id {id_max})')
print(f'NaN volumes     : {n_nan}')

### 4.4 Export

Writes `csv/02_stl_volumes.csv` and sets `stl_ok = True` only when the STL file exists on disk, its volume is at least `STL_OK_MIN_L`, and the mesh is a single connected solid (one shell, no detached junk shells).


In [ ]:
out = results_df[['config_id', 'volume_L', 'volume_m3']]
out.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(out)} rows to {OUTPUT_CSV}')

final_df = pd.read_csv(OUTPUT_CSV)

file_exists_values = []
for config_id in final_df['config_id']:
    file_exists_values.append(stl_path_for_config(config_id).exists())
file_exists = pd.Series(file_exists_values, index=final_df.index)
volume_ok = final_df['volume_L'].notna() & (final_df['volume_L'] >= STL_OK_MIN_L)

candidate_mask = file_exists & volume_ok
candidate_ids = final_df.loc[candidate_mask, 'config_id'].astype(int).tolist()
candidate_paths = []
for config_id in candidate_ids:
    candidate_paths.append(str(stl_path_for_config(config_id)))

single_solid_by_id = {}
worker_count = min(os.cpu_count() or 1, 16)
with ThreadPoolExecutor(max_workers=worker_count) as pool:
    results_iter = pool.map(stl_is_single_solid, candidate_paths)
    for config_id, is_solid in tqdm(zip(candidate_ids, results_iter), total=len(candidate_ids), desc=f'Validating STLs (workers={worker_count})'):
        single_solid_by_id[config_id] = bool(is_solid)

single_solid = final_df['config_id'].astype(int).map(single_solid_by_id).fillna(False)

final_df['stl_ok'] = file_exists & volume_ok & single_solid
final_df['single_solid'] = single_solid
final_df.to_csv(OUTPUT_CSV, index=False)

n_ok = int(final_df['stl_ok'].sum())
n_bad = len(final_df) - n_ok
invalid_ids = final_df.loc[~final_df['stl_ok'], 'config_id'].astype(int).tolist()
rejected_multi_shell = final_df.loc[candidate_mask & ~single_solid, 'config_id'].astype(int).tolist()

print(f'stl_ok: {n_ok} valid, {n_bad} invalid')
print(f'  rejected as broken geometry (multi-shell): {len(rejected_multi_shell)} -> {rejected_multi_shell[:50]}')
print(f'Invalid configs (all reasons): {", ".join(map(str, invalid_ids[:80]))}')

vols = final_df.loc[final_df['stl_ok'], 'volume_L']
n_vol = len(vols)
n_nan = final_df['volume_L'].isna().sum()
mean_val = vols.mean()
id_min = int(final_df.loc[vols.idxmin(), 'config_id'])
id_max = int(final_df.loc[vols.idxmax(), 'config_id'])
id_mean = int(final_df.loc[(vols - mean_val).abs().idxmin(), 'config_id'])

print(f'Volumes computed: {n_vol}/{len(final_df)}')
print(f'  mean  {mean_val:.4f} L  (config_id {id_mean})')
print(f'  min   {vols.min():.4f} L  (config_id {id_min})')
print(f'  max   {vols.max():.4f} L  (config_id {id_max})')
print(f'NaN volumes     : {n_nan}')

final_df.head()